# Řídké matice (Sparse Matrices)

Notebook pokrývá základy efektivního ukládání dat pomocí řídkých matic v Pythonu.

Knihovny: **NumPy**, **SciPy**

In [1]:
!pip install numpy scipy

## Husté matice (Dense matrices)

Hustá matice je dvourozměrné pole, kde jsou v paměti uloženy **všechny prvky** včetně nul.
To znamená, že hustá matice může být jednodušší na práci, protože nemusíme řešit vzor řídkosti,
ale pro velké matice s mnoha nulami je to plýtvání pamětí.

Matematicky lze hustou matici A o rozměrech m × n zapsat jako:

![Hustá matice](../img/dense.png)

Husté matice jsou běžně používány v lineární algebře, statistice, strojovém učení
a numerické analýze.

In [2]:
import numpy as np

In [3]:
matrix = np.array([[1, 0, 2, 0, 3],
                   [0, 0, 4, 0, 0],
                   [5, 0, 6, 0, 7],
                   [0, 0, 8, 0, 0],
                   [9, 0, 10, 0, 11]])
matrix

array([[ 1,  0,  2,  0,  3],
       [ 0,  0,  4,  0,  0],
       [ 5,  0,  6,  0,  7],
       [ 0,  0,  8,  0,  0],
       [ 9,  0, 10,  0, 11]])

## Řídké matice (Sparse matrices)

Řídké matice jsou specializovaná datová struktura pro matice, kde je **většina prvků nulových**.
Na rozdíl od hustých matic ukládají pouze nenulové prvky spolu s jejich indexy.
To může výrazně snížit paměťové nároky a zrychlit výpočty.

Existují různé reprezentace řídkých matic:

1. **COO (Coordinate List)** — ukládá nenulové prvky spolu s jejich řádkovými a sloupcovými indexy
2. **CSR (Compressed Sparse Row)** — optimalizováno pro řádkové operace
3. **CSC (Compressed Sparse Column)** — optimalizováno pro sloupcové operace

### Formát COO (Coordinate List)

Ukládá tři pole:
- `data` — hodnoty nenulových prvků
- `row` — řádkové indexy
- `col` — sloupcové indexy

In [4]:
from scipy.sparse import coo_matrix

matrix_coo = coo_matrix(matrix)
matrix_coo.data, matrix_coo.row, matrix_coo.col

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]),
 array([0, 0, 0, 1, 2, 2, 2, 3, 4, 4, 4], dtype=int32),
 array([0, 2, 4, 2, 0, 2, 4, 2, 0, 2, 4], dtype=int32))

Každý nenulový prvek je reprezentován trojicí `(row, col, data)`.
Např. prvek s hodnotou `4` se nachází na řádku `1`, sloupci `2`.

### Formát CSR (Compressed Sparse Row)

Ukládá tři pole:
- `data` — hodnoty nenulových prvků
- `indices` — sloupcové indexy nenulových prvků
- `indptr` — ukazatele na začátek každého řádku v poli `data`

Pole `indptr` má délku `počet_řádků + 1`.
Nenulové prvky řádku `i` najdeme v `data[indptr[i]:indptr[i+1]]`.

In [5]:
from scipy.sparse import csr_matrix

matrix_csr = csr_matrix(matrix)
matrix_csr.data, matrix_csr.indices, matrix_csr.indptr

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]),
 array([0, 2, 4, 2, 0, 2, 4, 2, 0, 2, 4], dtype=int32),
 array([ 0,  3,  4,  7,  8, 11], dtype=int32))

Např. pro řádek 0: `data[0:3]` = `[1, 2, 3]` na sloupcích `indices[0:3]` = `[0, 2, 4]`.

### Formát CSC (Compressed Sparse Column)

Analogický k CSR, ale komprimuje sloupcové informace místo řádkových.
Optimalizován pro sloupcové operace.

In [6]:
from scipy.sparse import csc_matrix

matrix_csc = csc_matrix(matrix)
matrix_csc.data, matrix_csc.indices, matrix_csc.indptr

(array([ 1,  5,  9,  2,  4,  6,  8, 10,  3,  7, 11]),
 array([0, 2, 4, 0, 1, 2, 3, 4, 0, 2, 4], dtype=int32),
 array([ 0,  3,  3,  8,  8, 11], dtype=int32))

Sloupec 0 obsahuje `data[0:3]` = `[1, 5, 9]` na řádcích `indices[0:3]` = `[0, 2, 4]`.

Sloupce 1 a 3 jsou prázdné — `indptr[1] == indptr[2]` a `indptr[3] == indptr[4]`.

### Porovnání paměťové náročnosti

Vytvořme velkou matici 10 000 × 10 000 s přibližně 90 % nulových prvků.

In [7]:
# Create a dense 10,000 × 10,000 matrix
dense_matrix = np.random.rand(10_000, 10_000)

# Set 90% of elements to zero
dense_matrix[dense_matrix < 0.9] = 0

In [8]:
# Ratio of non-zero elements (sparsity)
(dense_matrix > 0).mean()

np.float64(0.10002747)

In [9]:
# Number of zero elements
(dense_matrix == 0).sum()

np.int64(89997253)

In [10]:
# Memory usage of the dense matrix
f'Dense matrix: {dense_matrix.nbytes / 1e6} MB'

'Dense matrix: 800.0 MB'

In [11]:
def get_coo_matrix_size(sparse_matrix):
    size_of_data = sparse_matrix.data.nbytes
    size_of_rows = sparse_matrix.row.nbytes
    size_of_cols = sparse_matrix.col.nbytes
    return size_of_data + size_of_rows + size_of_cols

def get_csrcsc_matrix_size(sparse_matrix):
    size_of_data = sparse_matrix.data.nbytes
    size_of_indices = sparse_matrix.indices.nbytes
    size_of_indptr = sparse_matrix.indptr.nbytes
    return size_of_data + size_of_indices + size_of_indptr

In [12]:
dense_matrix_coo = coo_matrix(dense_matrix)
f'COO format: {get_coo_matrix_size(dense_matrix_coo) / 1e6} MB'

'COO format: 160.043952 MB'

In [13]:
dense_matrix_csr = csr_matrix(dense_matrix)
f'CSR format: {get_csrcsc_matrix_size(dense_matrix_csr) / 1e6} MB'

'CSR format: 120.072968 MB'

In [14]:
dense_matrix_csc = csc_matrix(dense_matrix)
f'CSC format: {get_csrcsc_matrix_size(dense_matrix_csc) / 1e6} MB'

'CSC format: 120.072968 MB'

**Proč formát COO zabírá více paměti než CSR a CSC?**

COO ukládá pro každý nenulový prvek zvlášť řádkový i sloupcový index.
CSR a CSC komprimují jednu dimenzi pomocí pole `indptr`,
které má délku pouze `počet_řádků + 1` (resp. `počet_sloupců + 1`).